# Imports, Path Setup & Load Model

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

# Determine repo root directory (two levels up from notebooks/isabel/)
REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == "isabel" else Path.cwd()

# Path to saved Keras model inside the artifacts directory
MODEL_PATH = REPO_ROOT / "artifacts" / "gluten_guard_efficientnet_full.keras"

# Directory containing test images categorized in subfolders
DATA_DIR = Path(
    "/home/isabelksommerfeld/code/lrnzgll/gluten-guard/data/"
    "food-101-predict-images-40-classes/food-101-predict-images-40-classes/"
)

# Validate that paths exist
assert MODEL_PATH.exists(), f"Model file not found at: {MODEL_PATH}"
assert DATA_DIR.exists(), f"Data directory not found at: {DATA_DIR}"

# Load trained EfficientNet model
print(f"Loading model from: {MODEL_PATH}")
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully!")

# Infer Class Names & Define Prediction Function

In [ ]:
# Derive class names alphabetically from directory names (matches image_dataset_from_directory ordering)
class_names = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Detected {len(class_names)} classes.")

def predict_image(image_path: Path):
    """
    Load and preprocess an image, then run model inference.
    Note: EfficientNet's preprocess_input layer is embedded in the model,
    so input images must remain in standard [0, 255] float32 scale (no /255 rescaling).
    """
    img = Image.open(image_path).convert("RGB").resize((224, 224))
    img_array = np.array(img, dtype=np.float32)
    img_batch = np.expand_dims(img_array, axis=0)  # Shape: (1, 224, 224, 3)

    # Run prediction
    probabilities = model.predict(img_batch, verbose=0)[0]
    top_index = np.argmax(probabilities)

    predicted_class = class_names[top_index]
    confidence = probabilities[top_index]

    return img, predicted_class, confidence

# Test Predictions & Display Results

In [ ]:
# OCalculate overall top-1 accuracy across all test files in all subfolders
all_images = list(DATA_DIR.rglob("*.jpg")) + list(DATA_DIR.rglob("*.png"))
correct = 0

for img_path in all_images:
    _, pred_class, _ = predict_image(img_path)
    true_class = img_path.parent.name
    if pred_class == true_class:
        correct += 1

total = len(all_images)
if total > 0:
    print(f"Overall Top-1 Accuracy: {correct}/{total} ({correct / total:.1%})")